# Letta / MemGPT (agent memory)

A self-contained refresher on giving LLM agents **persistent, self-editing long-term memory**.

**Domain:** Agentic AI  ·  *recommended addition*  ·  **runnable:** yes

## 1. What & Why

A vanilla LLM is **stateless and amnesiac**: it only knows what fits in its context window, and the moment a fact scrolls out of that window it's gone forever. Long conversations overflow; multi-session assistants forget who you are between visits.

**MemGPT** (the 2023 paper *"MemGPT: Towards LLMs as Operating Systems"*) reframes this as a **memory-management problem** and borrows the trick an OS uses for virtual memory: give the model a small fixed **in-context working memory** plus large **external storage**, and let the *agent itself* decide what to page in and out using tool/function calls. The model effectively gets unbounded, persistent memory on top of a finite context window.

**Letta** is the open-source framework (and hosted service) that productionizes MemGPT. It runs **stateful agents** as DB-backed server objects you talk to over a REST/SDK, plus an Agent Development Environment (ADE) for inspecting their memory. (MemGPT = the algorithm/paper; Letta = the framework. The old `pymemgpt` package is deprecated — use `letta`.)

**Reach for it when** you need an agent that *remembers across turns and sessions* — a personal assistant, a long-running support/tutor bot, a research agent accumulating findings.

**Don't bother when** the task is one-shot/stateless, or when the relevant knowledge is a static corpus you can serve with plain RAG, or when everything genuinely fits in a long-context model for a single call.

## 2. Mental Model

> **The LLM is a computer; the agent is the OS managing its memory.**

| OS concept | MemGPT/Letta equivalent |
|---|---|
| RAM — fast, tiny, expensive | **Context window** (system prompt + memory blocks + recent messages) |
| Disk — slow, huge, cheap | **External stores** (archival vector DB, recall log) |
| Process issuing syscalls | The **agent** calling memory-editing tools |
| Page fault → load from disk | `archival_memory_search` pulls old facts back into context |
| Eviction when RAM is full | Summarize/move overflow out of core memory to archival |

The defining move is **self-editing memory**: when you tell the agent "my name is Dana," it doesn't just reply — it *calls a function* (`core_memory_append`) to write that fact into its always-in-context memory so it persists. The model is both the user of memory and its manager.

## 3. Key Concepts

- **Context window / main context** — everything in-context this turn: system prompt, memory blocks, recent messages, tool definitions. Finite and the scarce resource.
- **Core memory** — small, *always-in-context*, **editable** blocks the agent maintains about itself and the world. Canonical blocks: `persona` (who the agent is) and `human` (what it knows about the user). In Letta blocks are labeled, size-limited, and can be **shared across agents**.
- **Recall memory** — a searchable log of the **full conversation history**, kept out of context and paged in on demand.
- **Archival memory** — an external **vector store** for arbitrary facts/documents; accessed via semantic search. Unbounded.
- **Memory-editing tools** — the functions the agent calls to manage memory: `core_memory_append`, `core_memory_replace`, `archival_memory_insert`, `archival_memory_search`.
- **Heartbeats / chained steps** — a tool call can request *another* reasoning step, so the agent can read → think → write within one turn instead of stopping after one action.
- **Context overflow → eviction** — when main context fills, older messages are recursively summarized and moved to recall/archival, freeing room.
- **Stateful agents** — Letta agents are persistent server-side objects (DB-backed), addressed by `agent_id`; "agents as a service." The **ADE** is the GUI for inspecting/editing their memory.

## 4. Setup

```bash
pip install letta          # full server + client (run `letta server`, or use Letta Cloud)
pip install letta-client   # just the Python SDK, to talk to a running server / cloud
```

A Letta agent needs an **LLM backend** (OpenAI/Anthropic API key, or a local model via Ollama) and an **embedding model** for archival search. Self-host with `letta server` or sign up for Letta Cloud and set `LETTA_API_KEY`.

The notebook below runs **without any of that**: Example 1 is a tiny pure-Python simulation of the memory hierarchy (stdlib only). Example 2 shows the real Letta SDK call shape but is **gated behind an env-var check**, so the notebook executes top-to-bottom even with no key.

In [ ]:
# Environment check — Example 1 needs only the stdlib; Example 2 needs a Letta/OpenAI key.
import importlib.util, os

has_letta = importlib.util.find_spec("letta_client") is not None
has_key = bool(os.getenv("LETTA_API_KEY") or os.getenv("OPENAI_API_KEY"))

print("letta-client installed:", has_letta)
print("API key available:    ", has_key)
print("Example 1 (simulation) runs regardless.")

## 5. Worked Examples

### Example 1 — Simulating the MemGPT memory hierarchy (pure Python, no deps)

We model the two ideas that make MemGPT work: **core memory** (small, always in context, self-edited) and **archival memory** (unbounded external store, searched on demand) — plus the **budget enforcement** that triggers eviction.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class MemoryBlock:
    """A labeled, size-limited slice of always-in-context core memory."""
    label: str
    value: str = ""
    limit: int = 200  # character budget — stands in for a token budget

    def append(self, text: str) -> None:
        self._commit((self.value + "\n" + text).strip())

    def replace(self, old: str, new: str) -> None:
        self._commit(self.value.replace(old, new))

    def _commit(self, candidate: str) -> None:
        # Reject the edit if it would blow the budget — the block stays unchanged,
        # so the caller can decide to evict to archival instead.
        if len(candidate) > self.limit:
            raise ValueError(
                f"core block '{self.label}' over budget "
                f"({len(candidate)}/{self.limit}) — must evict to archival"
            )
        self.value = candidate


class MiniMemGPT:
    """A toy agent exposing the memory-editing 'tools' a real MemGPT agent calls."""

    def __init__(self):
        self.core = {
            "persona": MemoryBlock("persona", "I am a patient coding tutor."),
            "human": MemoryBlock("human", ""),
        }
        self.archival: list[str] = []  # external store (stands in for a vector DB)

    # --- tools the LLM would invoke via function calling ---
    def core_memory_append(self, label, text):
        self.core[label].append(text)

    def core_memory_replace(self, label, old, new):
        self.core[label].replace(old, new)

    def archival_memory_insert(self, text):
        self.archival.append(text)

    def archival_memory_search(self, query, k=2):
        q = set(query.lower().split())
        scored = sorted(self.archival,
                        key=lambda d: len(q & set(d.lower().split())),
                        reverse=True)
        return [d for d in scored if q & set(d.lower().split())][:k]

    def context_window(self) -> str:
        """What the LLM actually sees each turn (core memory is always present)."""
        return "\n".join(f"[{b.label}] {b.value}" for b in self.core.values())


agent = MiniMemGPT()
print(agent.context_window())

When the user reveals durable facts, the agent **self-edits core memory** so they stay in context every future turn:

In [ ]:
# User: "Hi, I'm Dana and I'm learning Rust."  -> agent saves it to core memory
agent.core_memory_append("human", "Name: Dana.")
agent.core_memory_append("human", "Goal: learning Rust.")

# Later the goal sharpens -> in-place edit rather than a duplicate
agent.core_memory_replace("human", "Goal: learning Rust.", "Goal: ship a Rust CLI by Q3.")

print(agent.context_window())

Facts that are useful but don't need to sit in the tiny context window go to **archival memory**, retrieved later by semantic-ish search (here a toy keyword overlap stands in for embeddings):

In [ ]:
for note in [
    "Dana prefers worked examples over theory.",
    "Dana works in timezone UTC-6.",
    "Dana finished the borrow-checker chapter on 2026-06-10.",
    "Dana hit a lifetime error in her CLI argument parser.",
]:
    agent.archival_memory_insert(note)

print("Archival entries:", len(agent.archival))
print("Recall on 'borrow lifetime error':")
for hit in agent.archival_memory_search("borrow lifetime error"):
    print("  -", hit)

Finally, the **page-fault / eviction** behavior: writing past a core block's budget raises, and the OS-style response is to move the overflow to archival instead of crashing.

In [ ]:
try:
    agent.core_memory_append("human", "verbose backstory " * 30)
except ValueError as e:
    print("Page fault:", e)
    agent.archival_memory_insert("verbose backstory " * 30)  # spill to disk
    print("-> evicted overflow to archival; core memory stays within budget.")

print("\nFinal context window the LLM sees:")
print(agent.context_window())

### Example 2 — The real Letta SDK (gated behind an env-var check)

This is the actual call shape with `letta-client`. It only executes if a key is present, so the notebook still runs end-to-end without one. Note you never manually stuff memory — you create the agent with initial blocks and **the agent edits them itself** as the conversation unfolds.

In [ ]:
import os

if has_letta and has_key:
    from letta_client import Letta

    # Letta Cloud: token=...   |  self-host: base_url="http://localhost:8283"
    client = Letta(token=os.environ["LETTA_API_KEY"])

    agent_state = client.agents.create(
        memory_blocks=[
            {"label": "persona", "value": "I am a patient coding tutor."},
            {"label": "human", "value": "Name: Dana. Goal: learning Rust."},
        ],
        model="openai/gpt-4o-mini",
        embedding="openai/text-embedding-3-small",
    )

    response = client.agents.messages.create(
        agent_id=agent_state.id,
        messages=[{"role": "user",
                   "content": "Remember I prefer terse answers. What am I learning?"}],
    )
    for msg in response.messages:
        print(msg)

    # Inspect the (possibly self-edited) core memory blocks
    for block in client.agents.blocks.list(agent_id=agent_state.id):
        print(block.label, "->", block.value)
else:
    print("No Letta/OpenAI key set — skipping live call.")
    print("Call shape: client.agents.create(memory_blocks=[...], model=..., embedding=...)")
    print("            client.agents.messages.create(agent_id=..., messages=[...])")
    print("            client.agents.blocks.list(agent_id=...)  # inspect self-edited memory")

## 6. Gotchas & Pitfalls

- **Core memory is tiny on purpose.** Stuffing blocks crowds out the actual conversation. Keep only durable, high-value facts in core; everything else goes to archival.
- **Memory doesn't manage itself without the loop.** The agent only remembers if it's prompted and tooled to call the memory functions — and if the model is good enough at function calling. Weak models forget to save, or hallucinate edits.
- **Archival search is semantic, not exact.** Phrasing and the embedding model decide what comes back; misses happen. It's not a substitute for a structured query over a real database.
- **Every memory edit costs an extra LLM round-trip.** Heartbeat chaining (read → think → write) multiplies tokens and latency versus a plain completion. Budget for it.
- **Agents are stateful server objects.** State lives in the Letta DB keyed by `agent_id`; mixing up IDs, forgetting to persist, or never cleaning up test agents bites you.
- **Don't use it as a single-document context extender.** A 300-page PDF is better handled by RAG or a long-context model. MemGPT shines for *evolving, conversational* state, not one big static blob.
- **Naming/versioning.** MemGPT (paper/algorithm) ≠ Letta (framework). The legacy `pymemgpt` package is deprecated; install `letta` / `letta-client`.

## 7. When to Use vs Alternatives

| Approach | What it gives you | When it wins |
|---|---|---|
| **Letta / MemGPT** | Self-editing, persistent memory + stateful agent runtime + ADE | Assistants that must remember a user/task **across sessions** |
| **Long-context model** (Claude/Gemini) | Lossless recall *within* one huge window | Single call over a lot of text; simplest when it fits — but resets each call and cost scales with length |
| **Plain RAG** | Retrieval from a **static** corpus | Q&A over fixed documents; no need for the agent to *write* memory |
| **Framework memory** ([[langchain]] / [[langgraph]] buffers & summaries) | Lightweight conversation memory you wire yourself | You already live in that framework and want simple history/summary memory with full control |
| **Mem0 / Zep** | Memory-**as-a-service** you bolt onto any agent | You want a drop-in memory layer without adopting a whole agent runtime |

Rule of thumb: **long context** for "fits in one call," **RAG** for "static knowledge," and **Letta/MemGPT** for "the agent needs to *accumulate and edit* what it knows over time." These compose — a Letta agent commonly uses RAG over its archival memory. See also [[langgraph]] for graph-level control of the agent loop.

## 8. Resources

- **MemGPT paper** — *MemGPT: Towards LLMs as Operating Systems* (Packer et al., 2023): <https://arxiv.org/abs/2310.08560>
- **Letta documentation** — concepts, memory, agents, SDK: <https://docs.letta.com>
- **Letta on GitHub** — source, examples, the server: <https://github.com/letta-ai/letta>
- **"MemGPT is now part of Letta"** — the rename/relationship explained: <https://www.letta.com/blog/memgpt-and-letta>
- **Agent Development Environment (ADE)** — inspect & edit agent memory: <https://docs.letta.com/agent-development-environment>

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
class MemoryOverflow(ValueError):
    pass


class Agent:
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE